# AOP-ProCon — Step 4 **(build v3)**: Train All Models

Trains the models on the **Step-3 v3 pair files** (S2 v3, fingerprint `47d9c2a2fff63eb1`). Built on the v2.2 anti-collapse recipe (tau = 0.2, uniformity 1.5, cosine LR decay, grad clip 1.0).

### What build v3 changes

| # | Change | Why |
|---|---|---|
| 1 | **Provenance gate.** The S2 fingerprint, the descriptor SHA-256, the ESM-2 650M/150M SHA-256 (all from `step2_manifest.json`) and the SHA-256 of each pair file (from `pairs_manifest.json`) must match, or the notebook stops. Every checkpoint stores them. | A model can no longer be trained or evaluated on a mix of files from different runs. |
| 2 | **Pairs are loaded per experiment mode** from `pairs/<mode>/` (Step 3 v4): `strict` is the **primary analysis** (all models 1–10), `extended` (candidate evidence included) and `strict_ge4` (strict and ≥ 4 aa) are sensitivity analyses (Models 11 and 12). Train/validation split = the `fold` column (fold 0 trains, fold 1 validates); nothing is dropped. The prototype initialisation and ARI use the fold-0 peptides **of the same mode**. | Filtering one all-peptide pair file after the fact does not give a real strict experiment (random control no longer count-matched, k-NN anchors lose partners, scaler/threshold fitted on removed peptides). Each mode has its own complete pair set. |
| 3 | **Descriptor scaler for the fusion model is fitted on the training fold only** (mechanism-active fold-0 peptides of the primary mode). | It used to be fitted on all 1,389 sequences, including fold 1 and GEN. |
| 4 | **Mechanism imbalance is an explicit design decision.** The primary pairs contain ~50 times more FRS than MC rows, and batches are single-mechanism, so MC gets few gradient steps. `Primary` keeps the natural rate, `Primary-Balanced` resamples MC rows (same number of steps), and the new **Model 13 `Primary-MechEqual`** gives MC as many (resampled) batches per epoch as FRS. | The effect of the imbalance is measured instead of ignored. |
| 5 | Descriptors are re-indexed **by sequence** instead of asserting equal row order; embeddings are still checked against the S2 order and, with the descriptors, against the SHA-256 recorded by Step 2. | Order bugs fail loudly, not silently. |

Everything from the previous build that was not listed above is unchanged, including the defect fixes below (they stay because the code they describe stays).

### Earlier fixes (kept)

| ID | Defect | Fix in this notebook |
|---|---|---|
| **N-12** | A stale S1 pair file could be fed to training without anyone noticing. | Section 0b hard-fails unless the pair files are S2 (`mechanism` column, no `tier` column, duals present in both mechanisms) — and now also checks the manifests. |
| **E-014** | Step 4 once rebuilt and overwrote Step 3's k-NN file. | k-NN pairs are **loaded**; this notebook **writes nothing to `data/processed/`** — asserted at the end by SHA-256. |
| **N-9** | "Primary-150M" was trained on k-NN pairs, so it was never a scale test. | Model 8 uses the primary pairs; the confounded variant is Model 9 `k-NN-150M`. |
| **N-10 / N-5** | The checkpoint guard scored a GEN prototype that was not the one saved. | GEN is recomputed first; `min_proto_dist_saved` is measured on the saved tensors; both numbers are reported. |
| **E-013** | ARI selection alone can pick a collapsed representation. | Model 10 `ARI-Checkpoint-Floored` applies a floor on `min_proto_dist_saved`. |
| **E-007** | tau = 0.2 / lambda = 5.0 deviate from the TAF. | Section 2a states the deviation and the sweep that demonstrates it. |
| **N-4** | "The recipe gives a flat plateau" was claimed for every model. | The collapse report prints "k of N models hold a plateau" and names the exceptions. |

**Models trained: 15** (10 as before on the primary `strict` pairs + 2 sensitivity models + 1 mechanism-weighting model + 2 selection-rule models). **Reads** `data/processed/` (read-only) and **writes only** to `results/`.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/AOP-ProCon"

In [ ]:
import copy, collections, os, hashlib, json, datetime
from importlib import metadata
import numpy as np, pandas as pd
import torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler

os.makedirs(f"{BASE}/results", exist_ok=True)
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = "cpu"

def _ver(pkg):
    try: return metadata.version(pkg)
    except Exception: return "n/a"
VERSIONS = {"python": __import__("sys").version.split()[0], "torch": torch.__version__, "numpy": np.__version__,
            "pandas": pd.__version__, "scikit-learn": _ver("scikit-learn")}
print("Device:", device, "|", VERSIONS)

def s2_fingerprint(s2_df):
    # Canonical content hash of S2 -- identical to Steps 1-3: sequence, labels, homology_group, lso_fold, sample_weight.
    lines = []
    for r in s2_df.sort_values("sequence").itertuples():
        lines.append("\t".join([r.sequence, r.labels,
                                "" if pd.isna(r.homology_group) else str(int(r.homology_group)),
                                "" if pd.isna(r.lso_fold) else str(int(r.lso_fold)),
                                f"{float(r.sample_weight):.1f}"]))
    return hashlib.md5("\n".join(lines).encode()).hexdigest()[:16]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

### CONFIG — the anti-collapse recipe

Unchanged from v2.2. `tau` and `lam` deviate from the TAF; Section 2a states why.


In [ ]:
CONFIG = dict(
    tau=0.2,                 # TAF says 0.07 -- see Section 2a (E-007)
    lam=5.0,                 # TAF says 0.1  -- see Section 2a (E-007)
    lr=1e-4,
    use_uniformity=True,
    uniformity_weight=1.5,
    lr_decay=True,
    grad_clip=1.0,
)
PROTO_FLOOR = 0.5            # checkpoint guard for the loss-selected models
ARI_FLOOR   = 0.3            # Model 10's floor on ARI selection (E-013)
print("CONFIG:", CONFIG)

---
## 0. Load data and build lookups


In [ ]:
PROC = f"{BASE}/data/processed"
EMB  = f"{BASE}/data/embeddings"
EXPECTED_S2_FINGERPRINT = "47d9c2a2fff63eb1"      # Step 1 build v3

aop_benchpos = pd.read_csv(f"{PROC}/AOP-BenchPos_S2.csv")
descriptors  = pd.read_csv(f"{PROC}/descriptors.csv")

need = {"sequence", "labels", "sample_weight", "lso_fold", "cluster_40", "homology_group", "strict_evidence", "homology_resolvable"}
assert need <= set(aop_benchpos.columns), f"S2 is missing {need - set(aop_benchpos.columns)} -> not a Step 1 build v3 file"
assert aop_benchpos["sequence"].is_unique and descriptors["sequence"].is_unique
assert set(aop_benchpos["sequence"]) == set(descriptors["sequence"]), "S2 and descriptors hold different sequences"
descriptors = descriptors.set_index("sequence").loc[aop_benchpos["sequence"]].reset_index()     # align by KEY, not by position

# provenance gate: everything must come from the same S2 / Step 2 run
S2_FP = s2_fingerprint(aop_benchpos)
m2 = json.load(open(f"{PROC}/step2_manifest.json"))
assert S2_FP == m2["s2_fingerprint"], f"S2 fingerprint {S2_FP} != Step 2 manifest {m2['s2_fingerprint']} -> re-run Step 2"
EMB_SHA = {}
for name, path in [("descriptors.csv", f"{PROC}/descriptors.csv"), ("esm2_650M.npy", f"{EMB}/esm2_650M.npy"), ("esm2_150M.npy", f"{EMB}/esm2_150M.npy")]:
    EMB_SHA[name] = sha256_file(path)
    assert EMB_SHA[name] == m2["sha256"][name], f"{name} differs from the file Step 2 recorded -> re-run Step 2"
print("S2 fingerprint:", S2_FP, "| descriptors, ESM-2 650M and 150M SHA-256 match step2_manifest.json")
pm = json.load(open(f"{PROC}/pairs_manifest.json"))
assert pm.get("build", "").startswith("v4"), "pairs_manifest.json is not from Step 3 build v4 (mode-specific pairs) -> re-run Step 3"
assert pm["s2_fingerprint"] == S2_FP, f"pairs_manifest S2 fingerprint {pm['s2_fingerprint']} != {S2_FP} -> re-run Step 3"
MODES = pm["modes"]; PRIMARY_MODE = pm["primary_mode"]
print("pair modes:", list(MODES), "| primary:", PRIMARY_MODE)
if S2_FP != EXPECTED_S2_FINGERPRINT:
    print("NOTE: not Step 1 build v3 (", EXPECTED_S2_FINGERPRINT, ") -- consistent with Step 2 but a different split from the one validated in the Step 1 report")

esm_embeddings = np.load(f"{EMB}/esm2_650M.npy")
with open(f"{EMB}/esm2_650M_sequences.txt") as f:
    esm_sequences = f.read().splitlines()
assert esm_embeddings.shape[0] == len(aop_benchpos) and (aop_benchpos["sequence"].values == np.array(esm_sequences)).all()
seq_to_emb = {s: e for s, e in zip(aop_benchpos["sequence"], esm_embeddings)}
print(f"seq_to_emb: {len(seq_to_emb)} seqs, dim={esm_embeddings.shape[1]}")

esm_embeddings_150m = np.load(f"{EMB}/esm2_150M.npy")
with open(f"{EMB}/esm2_150M_sequences.txt") as f:
    esm_sequences_150m = f.read().splitlines()
assert esm_embeddings_150m.shape[0] == len(aop_benchpos) and (aop_benchpos["sequence"].values == np.array(esm_sequences_150m)).all()
seq_to_emb_150m = {s: e for s, e in zip(aop_benchpos["sequence"], esm_embeddings_150m)}

descriptor_cols = [c for c in descriptors.columns if c != "sequence"]
assert len(descriptor_cols) == 14, f"expected 14 descriptors, got {len(descriptor_cols)}"

mech_active = aop_benchpos[aop_benchpos["labels"].str.contains("FRS|MC")].reset_index(drop=True)
assert mech_active["lso_fold"].notna().all() and mech_active["homology_group"].notna().all()
assert mech_active.groupby("sequence")["lso_fold"].nunique().max() == 1, "a sequence is in both folds"
assert mech_active.groupby("homology_group")["lso_fold"].nunique().max() == 1, "a homology group is in both folds"
seq_fold_lookup = {s: int(f) for s, f in zip(mech_active["sequence"], mech_active["lso_fold"])}
gen_seqs = aop_benchpos[aop_benchpos["labels"] == "GEN"]["sequence"].tolist()

def mode_fold0(mode):
    # fold-0 mechanism-active peptides that belong to this experiment mode (prototype initialisation and ARI use exactly these)
    d = mech_active[mech_active["lso_fold"] == 0]
    if MODES[mode]["strict"]: d = d[d["strict_evidence"].astype(bool)]
    if MODES[mode]["ge4"]:    d = d[d["homology_resolvable"].astype(bool)]
    return d.reset_index(drop=True)
POOLS = {mode: mode_fold0(mode) for mode in MODES}
PRIMARY_POOL = POOLS[PRIMARY_MODE]
print({m: len(p) for m, p in POOLS.items()})

# HP2-Fusion input: descriptor scaler fitted on the TRAINING fold only (primary-mode fold-0 peptides), applied to every sequence
fit_mask = aop_benchpos["sequence"].isin(set(PRIMARY_POOL["sequence"])).values
desc_scaler = StandardScaler().fit(descriptors[descriptor_cols].values[fit_mask])
descriptors_scaled = desc_scaler.transform(descriptors[descriptor_cols].values)
seq_to_fused_emb = {s: np.concatenate([seq_to_emb[s], descriptors_scaled[i]]) for i, s in enumerate(descriptors["sequence"])}
seq_to_weight = dict(zip(aop_benchpos["sequence"], aop_benchpos["sample_weight"]))
print("down-weighted (candidates, weight 0.5):", int((aop_benchpos["sample_weight"] < 1.0).sum()))
print("mech-active:", len(mech_active), "| primary-mode fold0 pool:", len(PRIMARY_POOL), "| GEN:", len(gen_seqs))

DIM_650, DIM_150 = esm_embeddings.shape[1], esm_embeddings_150m.shape[1]
DIM_FUSED = DIM_650 + len(descriptor_cols)
print(f"input dims -> 650M {DIM_650} | 150M {DIM_150} | fused {DIM_FUSED}")

---
## 0b. Load the Step-3 v4 pair files and **prove they are the ones Step 3 wrote**  *(N-12 gate + manifest check)*

Pair files live in `data/processed/pairs/<mode>/` for the three modes of `pairs_manifest.json`. Every file must carry the S2 fingerprint and the SHA-256 recorded by Step 3, have exactly the columns `seq1, seq2, mechanism, distance, fold`, contain only peptides of its own mode (strict pools contain no candidate-evidence peptide; `strict_ge4` contains no peptide shorter than 4 aa), keep both members of every pair inside the same fold, and — as the single observable that separates S2 pairing from S1 pairing — contain dual peptides in both the FRS and the MC pairs.

**k-NN pairs are loaded, never rebuilt** (E-014).

In [ ]:
PAIR_NAMES = ["positive_pairs", "knn_pairs_k8", "ablation_random_pairs", "ablation_physchem_only_pairs"]
PAIRS, PAIR_HASHES = {}, {}
for mode in MODES:
    PAIRS[mode] = {}
    for n in PAIR_NAMES:
        key = f"{mode}/{n}.csv"; path = f"{PROC}/pairs/{key}"
        PAIR_HASHES[key] = sha256_file(path)
        assert PAIR_HASHES[key] == pm["sha256"][key], f"{key} differs from the file Step 3 recorded -> re-run Step 3"
        PAIRS[mode][n] = pd.read_csv(path)
print("pair files: S2 fingerprint and SHA-256 agree with pairs_manifest.json (", len(PAIR_HASHES), "files )")

LAB  = dict(zip(aop_benchpos["sequence"], aop_benchpos["labels"]))
STRICT_OF = dict(zip(aop_benchpos["sequence"], aop_benchpos["strict_evidence"].astype(bool)))
DUAL = {s for s, l in LAB.items() if l == "FRS;MC"}
print("dual peptides in S2:", len(DUAL))

def assert_s2(df, name, mode, directed=False):
    assert list(df.columns) == ["seq1", "seq2", "mechanism", "distance", "fold"], f"{mode}/{name}: not a Step 3 v4 file (columns {list(df.columns)})"
    assert set(df.mechanism) <= {"FRS", "MC"}, f"{mode}/{name}: unexpected mechanism value"
    assert df.seq1.isin(seq_fold_lookup).all() and df.seq2.isin(seq_fold_lookup).all(), f"{mode}/{name}: a pair member is not a mechanism-active S2 sequence"
    assert (df.seq1.map(seq_fold_lookup) == df.fold).all() and (df.seq2.map(seq_fold_lookup) == df.fold).all(), f"{mode}/{name}: a pair is not inside its fold"
    if MODES[mode]["strict"]:
        assert df.seq1.map(STRICT_OF).all() and df.seq2.map(STRICT_OF).all(), f"{mode}/{name}: a non-strict peptide is present"
    if MODES[mode]["ge4"]:
        assert (df.seq1.str.len() >= 4).all() and (df.seq2.str.len() >= 4).all(), f"{mode}/{name}: a peptide shorter than 4 aa is present"
    for m in sorted(set(df.mechanism)):
        sub = df[df.mechanism == m]
        assert (sub.seq1.map(LAB).str.contains(m) & sub.seq2.map(LAB).str.contains(m)).all(), f"N-12: {mode}/{name}/{m} has a pair whose member lacks label {m}"
    seen = {m: set(df[df.mechanism == m].seq1) | set(df[df.mechanism == m].seq2) for m in ("FRS", "MC")}
    both = len(DUAL & seen["FRS"] & seen["MC"])
    assert both > 0, f"N-12: {mode}/{name} has NO dual peptide in both FRS and MC pairs -> S1 pairing (duals were split)"
    return both

print("\nS2 pair-file gate:")
for mode in MODES:
    for n in PAIR_NAMES[:3]:
        both = assert_s2(PAIRS[mode][n], n, mode)
        print(f"  {mode:<11} {n:<24} {len(PAIRS[mode][n]):>6} pairs | duals in BOTH mechanisms: {both:>3}/{len(DUAL)} | cross-fold pairs: 0")
print("\npair-file SHA-256 (first 12):", {k: v[:12] for k, v in list(PAIR_HASHES.items())[:4]}, "...")

### Fold-split the pair sets

Step 3 v4 builds every pair inside one fold, so the split is just the `fold` column: **fold 0 trains, fold 1 validates**, nothing is dropped (asserted). Models 1–10 use the primary mode's files; Model 11 uses the `extended` positive pairs; Model 12 the `strict_ge4` positive pairs. Each of those trains with the prototype/ARI pool of its own mode.

In [ ]:
def split_by_fold(df):
    assert set(df["fold"]) <= {0, 1}
    tr, va = df[df.fold == 0].reset_index(drop=True), df[df.fold == 1].reset_index(drop=True)
    assert len(tr) + len(va) == len(df)
    return tr, va

P0 = PAIRS[PRIMARY_MODE]
primary_train, primary_val = split_by_fold(P0["positive_pairs"])
hp1_train,     hp1_val     = split_by_fold(P0["ablation_random_pairs"])
knn_train,     knn_val     = split_by_fold(P0["knn_pairs_k8"])
ext_train,     ext_val     = split_by_fold(PAIRS["extended"]["positive_pairs"])
ge4_train,     ge4_val     = split_by_fold(PAIRS["strict_ge4"]["positive_pairs"])

for n, tr, va in [(f"primary ({PRIMARY_MODE})", primary_train, primary_val), ("hp1_random", hp1_train, hp1_val), ("knn_k8", knn_train, knn_val),
                  ("extended", ext_train, ext_val), ("strict_ge4", ge4_train, ge4_val)]:
    print(f"{n:<18} train={len(tr):>6} val={len(va):>6}   " + "   ".join(f"{m}: train={int((tr.mechanism == m).sum()):>5} val={int((va.mechanism == m).sum()):>5}" for m in ("FRS", "MC")))
    assert len(tr) > 0 and len(va) > 0, f"{n}: empty split"
    assert (tr.mechanism == "MC").sum() > 0 and (va.mechanism == "MC").sum() > 0, f"{n}: no MC pairs in train or validation"
n_f, n_m = (primary_train.mechanism == "FRS").sum(), (primary_train.mechanism == "MC").sum()
print(f"\nprimary training pairs: FRS : MC = {n_f} : {n_m} = {n_f / n_m:.1f} : 1  -> mechanism weighting is a design decision (Models 1, 4 and 13)")
print(f"primary MC validation pairs: {int((primary_val.mechanism == 'MC').sum())}  -> the MC part of the validation loss is very noisy; do not read it as evidence")

---
## 1. The honest post-split partner count

Step 3 v4 searches neighbours **inside the fold**, so the "every anchor has 8 partners" guarantee of the k-NN file holds for the training pairs themselves (asserted below). The **percentile** pairs have no such guarantee — their partner counts, including the anchors with no partner at all, are reported here so the thin MC signal is quoted with its numbers.

In [ ]:
def partner_report(tr, name):
    rows = []
    for m in ("FRS", "MC"):
        d = tr[tr.mechanism == m]
        deg = pd.concat([d.seq1, d.seq2]).value_counts()
        n_anchor = int((PRIMARY_POOL.labels.str.contains(m)).sum())
        rows.append({"set": name, "mech": m, "fold0_peptides": n_anchor, "with_partner": len(deg),
                     "no_partner_%": round(100 * (1 - len(deg) / n_anchor), 1),
                     "min": int(deg.min()), "median": float(deg.median()), "max": int(deg.max()), "lt8": int((deg < 8).sum())})
    return rows

post = pd.DataFrame(partner_report(primary_train, "primary") + partner_report(knn_train, "knn_k8"))
print(post.to_string(index=False))
for m in ("FRS", "MC"):
    out_deg = knn_train[knn_train.mechanism == m].seq1.value_counts()
    assert out_deg.min() == out_deg.max() == 8, f"k-NN {m}: an anchor does not have exactly 8 training partners"
print("\nk-NN: every fold-0 anchor has exactly 8 training partners (asserted).")
print("Percentile pairs: quote the no_partner_% column for MC -- the MC training signal is thin.")

---
## 2. Shared training functions


### 2a. Deviation from the TAF: `tau` and `lambda`  *(E-007)*

| Parameter | TAF | Used here | Why |
|---|---|---|---|
| `tau` (temperature) | 0.07 | **0.20** | measured necessity, below |
| `lambda` (push-away weight) | 0.1 | **5.0** | measured necessity, below |

**The mechanism.** The loss is *positive-only*: `-mean(cos(a, p) / tau)` with **no denominator and no
negatives**. In InfoNCE, `tau` scales a softmax and its effect is bounded. Here it scales a bare
alignment term, so the gradient magnitude goes as **1/tau**: at tau=0.07 the alignment gradient is
~14x its tau=1 size, and it overwhelms the `lambda`-weighted prototype push-away whatever `lambda`
is set to. The representation then collapses — every peptide maps to one point and all prototypes
merge.

**What the sweep below can and cannot show.** It plots two curves per recipe: the *guard* value (GEN frozen at its initial position) and the *saved* value (GEN recomputed from the trained head). The guard curve shows that the recipes behave differently over training. Whether a recipe produces a *separated prototype set in the checkpoint that would be saved* is read from the **saved** curve (right panel) and from the run-time verdict printed under the figure — not from this text. The numbers in the verdict, not the wording here, are what may be quoted.

**Framing.** The 1/tau gradient argument is an argument about the loss, and it is the reason for the deviation; it is reported as a
deviation in the thesis, not silently absorbed. Whether tau = 0.2 *works* is a separate, empirical question answered by the sweep verdict. The TAF's tau=0.07 is the standard SimCLR value,
which is correct for a loss *with* a denominator.

In [ ]:
def make_batches(df, batch_size=64, seed=None, mc_mode="natural"):
    rng = np.random.RandomState(seed); batches = []
    frs = df[df["mechanism"] == "FRS"].sample(frac=1, random_state=seed).reset_index(drop=True)
    for start in range(0, len(frs), batch_size):
        chunk = frs.iloc[start:start + batch_size]
        if len(chunk) > 0: batches.append((chunk, "FRS"))
    n_frs = len(batches)
    mc = df[df["mechanism"] == "MC"].reset_index(drop=True)
    if mc_mode == "equal" and len(mc) > 0 and n_frs > 0:
        # mechanism-equal steps: as many MC batches per epoch as FRS batches, MC rows resampled (inverse anchor-frequency weights)
        freq = collections.Counter(list(mc["seq1"]) + list(mc["seq2"]))
        w = np.array([1.0 / (freq[r.seq1] + freq[r.seq2]) for r in mc.itertuples()]); w = w / w.sum()
        idx_all = rng.choice(len(mc), size=n_frs * batch_size, replace=True, p=w)
        for start in range(0, len(idx_all), batch_size):
            batches.append((mc.iloc[idx_all[start:start + batch_size]], "MC"))
    elif mc_mode == "balanced" and len(mc) > 0 and n_frs > 0:
        freq = collections.Counter(list(mc["seq1"]) + list(mc["seq2"]))
        w = np.array([1.0 / (freq[r.seq1] + freq[r.seq2]) for r in mc.itertuples()]); w = w / w.sum()
        idx_all = rng.choice(len(mc), size=len(mc), replace=True, p=w)
        for start in range(0, len(idx_all), batch_size):
            idx = idx_all[start:start + batch_size]
            if len(idx) > 0: batches.append((mc.iloc[idx], "MC"))
    else:
        for start in range(0, len(mc), batch_size):
            chunk = mc.iloc[start:start + batch_size]
            if len(chunk) > 0: batches.append((chunk, "MC"))
    rng.shuffle(batches); return batches

def positive_only_loss(anchor, positive, weights=None, tau=0.07):
    anchor = F.normalize(anchor, dim=-1); positive = F.normalize(positive, dim=-1)
    sim = (anchor * positive).sum(-1) / tau
    if weights is None: return -sim.mean()
    w = weights / weights.sum(); return -(sim * w).sum()

def uniformity_loss(x):
    # Wang & Isola (2020). x: normalized [N,D]. Minimising it spreads points on the hypersphere.
    if x.shape[0] < 2: return torch.tensor(0.0, device=x.device)
    sq = torch.pdist(x, p=2).pow(2)
    return sq.mul(-2.0).exp().mean().log()

class Prototypes:
    def __init__(self, init_vectors, dynamic, ema=0.99):
        self.vectors = {k: F.normalize(v, dim=-1) for k, v in init_vectors.items()}
        self.dynamic = dynamic; self.ema = ema
    def update(self, mech, batch_mean):
        if mech not in self.dynamic: return
        self.vectors[mech] = F.normalize(self.ema * self.vectors[mech] + (1 - self.ema) * batch_mean, dim=-1)
    def push_away_loss(self, emb, mech, lam=5.0):
        other = [v for k, v in self.vectors.items() if k != mech]
        sim = sum(F.cosine_similarity(emb, o.unsqueeze(0).expand_as(emb)).mean() for o in other)
        return lam * sim
    def pairwise_distance_report(self):
        ks = list(self.vectors.keys())
        return {f"{ks[i]}-{ks[j]}": (1 - (self.vectors[ks[i]] * self.vectors[ks[j]]).sum()).item()
                for i in range(len(ks)) for j in range(i + 1, len(ks))}

def proto_distances(d):
    ks = sorted(d.keys())
    return {f"{ks[i]}-{ks[j]}": float(1 - (F.normalize(d[ks[i]], dim=-1) *
                                           F.normalize(d[ks[j]], dim=-1)).sum())
            for i in range(len(ks)) for j in range(i + 1, len(ks))}

### 2b. `min_proto_dist` is measured on the tensors that actually get saved  *(N-10 fix)*

**The defect.** `Prototypes` is constructed with `dynamic={"FRS","MC"}`, so the GEN vector is frozen
at its pre-training value for the whole run. The checkpoint guard then scored
`{FRS_trained, MC_trained, GEN_pretraining}` and recorded `min_proto_dist`. But `save_model`
**replaced** GEN with a freshly computed trained-head mean immediately before `torch.save` — and never
re-measured. So a checkpoint could pass the 0.5 floor and still be written to disk with
FRS-GEN = 0.0001. That is exactly the contradiction between Step 4's "collapse-free" claim and
Step 5's prototype-distance table, and it is why the anti-collapse recipe appeared not to fix
FRS≈GEN: the guard was never looking at GEN.

**The fix.** `gen_trained_for` computes the post-training GEN vector *for any head state*, so
`min_proto_dist_saved` can be evaluated on the exact dictionary being serialised. Both numbers are
kept:

- `min_proto_dist_guard` — what the old code reported (FRS/MC trained + GEN frozen)
- `min_proto_dist_saved` — the truth about the saved prototypes

Model 10's floor is applied to `min_proto_dist_saved`.

**What a small saved FRS–GEN distance means.** GEN peptides are a pool of unknown mechanism, not a class the head is trained to separate from FRS (the loss never pushes GEN peptides anywhere; the GEN prototype is only a mean over them). If the head maps FRS and GEN peptides to the same region, the saved FRS–GEN distance is ≈ 0 **without any collapse of the representation**. So FRS–GEN ≈ 0 is reported as *descriptive* ("the head does not separate these two populations"), and collapse is judged from the spread diagnostics logged every epoch: `mean_cos` (mean pairwise cosine of the fold-0 mechanism-active embeddings; 1.0 = all peptides at one point) and `eff_rank` (number of embedding directions in use). No threshold is applied to them; they are reported next to the prototype distances.

In [ ]:
def train_model(train_pairs, val_pairs, embed_lookup, input_dim, run_name,
                mc_mode="natural", checkpoint_mode="val_loss",
                tau=0.07, lam=5.0, lr=1e-3, use_uniformity=False, uniformity_weight=1.0,
                lr_decay=False, grad_clip=None,
                max_epochs=40, patience=12, checkpoint_threshold=0.5,
                saved_dist_floor=None, seed=SEED, verbose=True, proto_pool=None):
    torch.manual_seed(seed); np.random.seed(seed)          # per-model reseed: models stay comparable
    pool = PRIMARY_POOL if proto_pool is None else proto_pool            # fold-0 peptides of the mode: prototype initialisation and ARI
    head = torch.nn.Linear(input_dim, 128).to(device)
    optimizer = torch.optim.Adam(head.parameters(), lr=lr)
    scheduler = (torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
                 if lr_decay else None)

    def embed_batch(chunk, col):
        return torch.tensor(np.stack([embed_lookup[s] for s in chunk[col]]),
                            dtype=torch.float32).to(device)
    def batch_weights(chunk):
        return torch.tensor([(seq_to_weight[a] + seq_to_weight[b]) / 2.0
                             for a, b in zip(chunk["seq1"], chunk["seq2"])],
                            dtype=torch.float32).to(device)
    def project(seqs):
        raw = torch.tensor(np.stack([embed_lookup[s] for s in seqs]), dtype=torch.float32).to(device)
        with torch.no_grad(): out = head(raw)
        return F.normalize(out, dim=-1)
    def epoch_ari():
        emb = project(pool["sequence"].tolist()).numpy()
        lab = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(emb)
        return float(adjusted_rand_score(pool["labels"].values, lab))

    def spread_stats():
        # collapse diagnostics on the mode's fold-0 mechanism-active peptides (no threshold is applied to them):
        #   mean_cos : mean pairwise cosine of the unit-norm embeddings (1.0 = every peptide at one point)
        #   eff_rank : exp(entropy of the normalised squared singular values of the centred embeddings) = number of directions in use
        z = project(pool["sequence"].tolist())
        mean_cos = float(((z @ z.T).sum() - len(z)) / (len(z) * (len(z) - 1)))
        s2 = torch.linalg.svdvals(z - z.mean(0, keepdim=True)).numpy() ** 2
        p = s2[s2 > 0] / s2.sum()
        return mean_cos, float(np.exp(-(p * np.log(p)).sum()))

    gen_raw = torch.tensor(np.stack([embed_lookup[s] for s in gen_seqs]),
                           dtype=torch.float32).to(device)
    def gen_trained_for(head_state):            # N-10: the GEN vector that WOULD be saved
        probe = torch.nn.Linear(input_dim, 128).to(device)
        probe.load_state_dict(head_state)
        with torch.no_grad():
            return F.normalize(probe(gen_raw).mean(0), dim=-1)
    def mech_centroid(mech):
        seqs = (pool[pool["labels"].str.contains(mech)]["sequence"].unique().tolist())
        raw = torch.tensor(np.stack([embed_lookup[s] for s in seqs]), dtype=torch.float32).to(device)
        with torch.no_grad(): return head(raw).mean(0)

    with torch.no_grad(): gen_init = head(gen_raw).mean(0)
    prototypes = Prototypes({"FRS": mech_centroid("FRS"), "MC": mech_centroid("MC"), "GEN": gen_init},
                            dynamic={"FRS", "MC"}, ema=0.99)

    def run_batch(chunk, mech, train=True):
        a_raw = embed_batch(chunk, "seq1"); p_raw = embed_batch(chunk, "seq2")
        if train:
            a = head(a_raw); p = head(p_raw)
        else:
            with torch.no_grad(): a = head(a_raw); p = head(p_raw)
        loss = (positive_only_loss(a, p, weights=batch_weights(chunk), tau=tau)
                + prototypes.push_away_loss(a, mech, lam=lam))
        if use_uniformity:
            loss = loss + uniformity_weight * uniformity_loss(F.normalize(a, dim=-1))
        return loss, a

    best_state, best_train_safe, best_ari, since = None, float("inf"), -1.0, 0
    best_cos = float("inf")
    fallback_state, fallback_dist = None, -1.0
    history = {"epoch": [], "train_loss": [], "val_loss": [], "proto_dist": [],
               "proto_dist_saved": [], "ari": [], "mean_cos": [], "eff_rank": []}

    for epoch in range(max_epochs):
        head.train(); tl = []
        for chunk, mech in make_batches(train_pairs, 64, seed=epoch, mc_mode=mc_mode):
            loss, a = run_batch(chunk, mech, True)
            optimizer.zero_grad(); loss.backward()
            if grad_clip is not None: torch.nn.utils.clip_grad_norm_(head.parameters(), grad_clip)
            optimizer.step()
            prototypes.update(mech, a.mean(0).detach()); tl.append(loss.item())
        head.eval(); vl = []
        for chunk, mech in make_batches(val_pairs, 64, seed=1000 + epoch, mc_mode="natural"):
            loss, _ = run_batch(chunk, mech, False); vl.append(loss.item())
        train_loss = float(np.mean(tl)); val_loss = float(np.mean(vl)) if vl else float("nan")
        if scheduler is not None: scheduler.step()

        min_dist = min(prototypes.pairwise_distance_report().values())        # old guard value
        cand = {k: v.clone() for k, v in prototypes.vectors.items()}          # N-10: what gets saved
        cand["GEN"] = gen_trained_for(head.state_dict())
        saved_d = proto_distances(cand)
        min_dist_saved = min(saved_d.values())
        mean_cos, eff_rank = spread_stats()

        cur_ari = epoch_ari() if checkpoint_mode == "ari" else float("nan")
        for k, v in zip(["epoch", "train_loss", "val_loss", "proto_dist", "proto_dist_saved", "ari", "mean_cos", "eff_rank"],
                        [epoch, train_loss, val_loss, min_dist, min_dist_saved, cur_ari, mean_cos, eff_rank]):
            history[k].append(v)

        snap = lambda: {"head": copy.deepcopy(head.state_dict()),
                        "prototypes": {k: v.clone() for k, v in prototypes.vectors.items()},
                        "prototypes_saved": {k: v.clone() for k, v in cand.items()},
                        "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                        "min_proto_dist": min_dist, "min_proto_dist_saved": min_dist_saved,
                        "saved_distances": saved_d, "ari": cur_ari, "mean_cos": mean_cos, "eff_rank": eff_rank}

        if min_dist_saved > fallback_dist:          # best-separated epoch, criterion aside
            fallback_dist, fallback_state = min_dist_saved, snap()

        improved = False
        if checkpoint_mode == "ari":
            # E-013: optional floor on the SAVED prototypes, so ARI cannot select a collapsed state
            floor_ok = (saved_dist_floor is None) or (min_dist_saved > saved_dist_floor)
            if floor_ok and cur_ari > best_ari:
                best_ari = cur_ari; best_state = snap(); improved = True
        elif checkpoint_mode == "spread":
            # keep the epoch whose embeddings are the most spread out (lowest mean pairwise cosine); no label and no loss involved
            if mean_cos < best_cos:
                best_cos = mean_cos; best_state = snap(); improved = True
        else:
            if min_dist > checkpoint_threshold and train_loss < best_train_safe:
                best_train_safe = train_loss; best_state = snap(); improved = True

        if np.isnan(train_loss):
            if verbose: print(f"  [{run_name}] Epoch {epoch}: NaN -- stopping.")
            break
        flag = " COLLAPSE" if min_dist < 0.1 else ""
        flag += " GEN-MERGED" if min_dist_saved < 0.1 <= min_dist else ""
        if verbose and (epoch % 5 == 0 or epoch < 6 or flag):
            print(f"  [{run_name}] Epoch {epoch:3d} | train={train_loss:.4f} | val={val_loss:.4f} "
                  f"| guard={min_dist:.4f} | saved={min_dist_saved:.4f} | mean_cos={mean_cos:.3f} eff_rank={eff_rank:.1f}{flag}")
        if improved: since = 0
        elif best_state is not None:
            since += 1
            if since >= patience:
                if verbose: print(f"  [{run_name}] Early stopping at epoch {epoch}")
                break

    criterion_met = best_state is not None
    if not criterion_met and fallback_state is not None:
        # Never abandon a whole 10-model run because one model failed the guard: keep the
        # epoch with the best SAVED prototype separation and flag it loudly instead.
        best_state = fallback_state
        if verbose:
            print(f"  [{run_name}] *** NO epoch met the selection criterion. Falling back to the "
                  f"best-separated epoch ({best_state['epoch']}, saved min_dist="
                  f"{best_state['min_proto_dist_saved']:.4f}). This model is NOT comparable to "
                  f"the others -- report it as 'criterion not met'.")
    if best_state is not None:
        best_state["criterion_met"] = criterion_met
        head.load_state_dict(best_state["head"]); prototypes.vectors = best_state["prototypes"]
        if verbose:
            print(f"  [{run_name}] Restored epoch {best_state['epoch']}: "
                  f"guard min_dist={best_state['min_proto_dist']:.4f}, "
                  f"SAVED min_dist={best_state['min_proto_dist_saved']:.4f}")
            if best_state["min_proto_dist_saved"] < 0.1:
                print(f"  [{run_name}] *** WARNING (N-10): saved prototypes are merged "
                      f"({best_state['saved_distances']}). The guard value above does NOT "
                      f"describe this checkpoint.")
    elif verbose:
        print(f"  [{run_name}] WARNING: no checkpoint at all (training produced nothing usable).")
    return {"head": head, "prototypes": prototypes, "history": history,
            "best_state": best_state, "embed_lookup": embed_lookup,
            "run_name": run_name, "input_dim": input_dim}

### 2c. `save_model` — writes the re-measured distances into the checkpoint

`prototypes_saved` was already built and measured during training, so saving is now just
serialisation: no prototype is modified after being scored. The pre-training GEN snapshot is kept
under `GEN_pretraining_snapshot` for comparison, and the pair-file SHA-256 values, the S2 fingerprint, the input-file hashes, the pair variant (`all` / `strict` / `ge4`), the seed and the package versions travel with the
checkpoint so Step 5 can confirm exactly what the model saw (N-12).

In [ ]:
SAVED = []

def save_model(run, filename, run_name, variant="all", mode=None):
    mode = PRIMARY_MODE if mode is None else mode
    bs = run["best_state"]
    if bs is None:
        print(f"  !! {run_name}: NOTHING TO SAVE (training produced no state). Skipping.")
        SAVED.append({"model": run_name, "file": None, "variant": variant, "epoch": None, "guard": None,
                      "saved": None, "FRS-GEN": None, "FRS-MC": None, "GEN-MC": None,
                      "ari_at_ckpt": None, "criterion_met": False})
        return
    protos = {k: v.cpu() for k, v in bs["prototypes_saved"].items()}
    protos["GEN_pretraining_snapshot"] = bs["prototypes"]["GEN"].cpu()
    d = proto_distances({k: v for k, v in protos.items() if k != "GEN_pretraining_snapshot"})
    assert abs(min(d.values()) - bs["min_proto_dist_saved"]) < 1e-5, "N-10: saved distances drifted"

    path = f"{BASE}/results/{filename}"
    torch.save({"head_state_dict": bs["head"], "prototypes": protos,
                "history": run["history"], "checkpoint_epoch": bs["epoch"],
                "input_dim": run["input_dim"], "gen_prototype_recomputed": True,
                "min_proto_dist_guard": bs["min_proto_dist"],
                "min_proto_dist_saved": bs["min_proto_dist_saved"],
                "mean_cos_at_ckpt": bs["mean_cos"], "eff_rank_at_ckpt": bs["eff_rank"],
                "saved_distances": d, "config": CONFIG, "pair_hashes": PAIR_HASHES,
                "s2_fingerprint": S2_FP, "input_sha256": EMB_SHA, "pair_variant": variant, "pair_mode": mode, "seed": SEED, "versions": VERSIONS,
                "criterion_met": bool(bs.get("criterion_met", True))}, path)
    assert os.path.exists(path)
    SAVED.append({"model": run_name, "file": filename, "variant": variant, "pair_mode": mode, "epoch": bs["epoch"],
                  "guard": round(bs["min_proto_dist"], 4),
                  "saved": round(bs["min_proto_dist_saved"], 4),
                  "FRS-GEN": round(d.get("FRS-GEN", float("nan")), 4),
                  "FRS-MC": round(d.get("FRS-MC", float("nan")), 4),
                  "GEN-MC": round(d.get("GEN-MC", float("nan")), 4),
                  "ari_at_ckpt": bs["ari"], "mean_cos": round(bs["mean_cos"], 4), "eff_rank": round(bs["eff_rank"], 2),
                  "criterion_met": bool(bs.get("criterion_met", True))})
    print(f"  Saved {filename}: epoch={bs['epoch']} guard={bs['min_proto_dist']:.3f} "
          f"SAVED={bs['min_proto_dist_saved']:.3f} | {d}")
    if d.get("FRS-GEN", 1.0) < 0.1:
        print("  *** FRS and GEN are merged in this saved model (N-5/N-10).")

---
## 3. Config sweep — including the exact TAF recipe  *(the evidence behind Section 2a)*

Four recipes on the Primary pairs, 30 epochs, nothing saved. The two TAF rows are what make this a
deviation *argument* rather than an assertion: the specification's own parameters are run and shown
to collapse. Both `min_proto_dist` curves are plotted — the old guard value **and** the saved value —
because they diverge, which is N-10 made visible.


In [ ]:
sweep_configs = [
    ("TAF: tau=0.07 lam=0.1",   dict(tau=0.07, lam=0.1, lr=1e-3, use_uniformity=False)),
    ("v2: tau=0.07 lam=5.0",    dict(tau=0.07, lam=5.0, lr=1e-3, use_uniformity=False)),
    ("v2.1 (unif, no decay)",   dict(tau=0.2, lam=5.0, lr=1e-4, use_uniformity=True,
                                     uniformity_weight=1.0)),
    ("v2.2 (unif 1.5 + decay)", dict(tau=0.2, lam=5.0, lr=1e-4, use_uniformity=True,
                                     uniformity_weight=1.5, lr_decay=True, grad_clip=1.0)),
]
sweep_hist = {}
for name, cfg in sweep_configs:
    run = train_model(primary_train, primary_val, seq_to_emb, DIM_650, name,
                      max_epochs=30, patience=30, verbose=False, **cfg)
    h = run["history"]; sweep_hist[name] = h
    print(f"{name:26s} | guard peak={max(h['proto_dist']):.3f} final={h['proto_dist'][-1]:.3f} "
          f"| SAVED peak={max(h['proto_dist_saved']):.3f} final={h['proto_dist_saved'][-1]:.3f} "
          f"| epochs SAVED>0.5: {sum(1 for d in h['proto_dist_saved'] if d > 0.5)}/{len(h['epoch'])}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for key, ax, title in [("proto_dist", axes[0], "guard value (GEN frozen) -- what v2.2 reported"),
                       ("proto_dist_saved", axes[1], "SAVED value (GEN recomputed) -- the truth (N-10)")]:
    for name, h in sweep_hist.items():
        ax.plot(h["epoch"], h[key], marker="o", ms=3, label=name)
    ax.axhline(0.5, ls="--", c="gray"); ax.axhline(0.1, ls=":", c="red")
    ax.set_xlabel("epoch"); ax.set_title(title, fontsize=10); ax.grid(alpha=0.3)
axes[0].set_ylabel("min prototype distance"); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{BASE}/results/anticollapse_sweep_fixed.png", dpi=120); plt.show()
print("\nSaved: results/anticollapse_sweep_fixed.png")
print("Left panel is the curve the thesis used; right panel is the one that describes the")
print("checkpoints actually written to disk. Where they differ, the left panel overstates.")

print("\n" + "=" * 70)
print("SWEEP VERDICT (computed, quote these numbers):")
n_sep = 0
for name, h in sweep_hist.items():
    peak_saved = max(h["proto_dist_saved"]); n_sep += int(peak_saved > 0.5)
    print(f"  {name:26s} saved peak={peak_saved:.3f}  saved final={h['proto_dist_saved'][-1]:.3f}  | mean_cos final={h['mean_cos'][-1]:.3f}  eff_rank final={h['eff_rank'][-1]:.1f}")
print(f"  {n_sep} of {len(sweep_hist)} recipes ever reach a SAVED min prototype distance > 0.5.")
if n_sep == 0:
    print("  -> No recipe, including v2.2, gives a saved prototype set with all three prototypes separated. The guard curve (left panel)")
    print("     must not be used to claim that tau = 0.2 'holds a separated plateau'. The FRS-GEN part of the saved distance is descriptive (Section 2b);")
    print("     judge collapse by mean_cos / eff_rank, and judge usefulness by Step 5 retrieval, not by this figure.")

---
## 4. Train all 15 models

| # | Model | Pairs | Embedding | Selection |
|---|---|---|---|---|
| 1 | Primary | percentile | 650M | loss, guard > 0.5 |
| 2 | HP1-Random | random | 650M | loss, guard > 0.5 |
| 3 | HP2-Fusion | percentile | 650M + 14-D | loss, guard > 0.5 |
| 4 | Primary-Balanced | percentile | 650M | loss, guard > 0.5 |
| 5 | ARI-Checkpoint-Only | percentile | 650M | ARI, **no floor** |
| 6 | k-NN-Pairing | k-NN K=8 | 650M | loss, guard > 0.5 |
| 7 | k-NN+ARI-Checkpoint | k-NN K=8 | 650M | ARI, **no floor** |
| 8 | **Primary-150M** *(N-9 fixed)* | **percentile** | 150M | loss, guard > 0.5 |
| 9 | **k-NN-150M** *(was mislabelled "Primary-150M")* | k-NN K=8 | 150M | loss, guard > 0.5 |
| 10 | **ARI-Checkpoint-Floored** *(E-013)* | percentile | 650M | ARI **and `saved > 0.3`** |
| 11 | **Primary-Extended** *(sensitivity)* | percentile, **extended** mode (candidate evidence included) | 650M | loss, guard > 0.5 |
| 12 | **Primary-StrictGE4** *(sensitivity)* | percentile, **strict_ge4** mode (strict and ≥ 4 aa) | 650M | loss, guard > 0.5 |
| 13 | **Primary-MechEqual** *(weighting)* | percentile, strict; MC resampled to as many batches per epoch as FRS | 650M | loss, guard > 0.5 |
| 14 | **Primary-MaxSpread** *(selection rule)* | percentile, strict (same as Model 1) | 650M | **epoch with the lowest mean pairwise cosine** (most spread-out embeddings) |
| 15 | **MechEqual-MaxSpread** *(weighting + selection)* | percentile, strict; MC resampled to as many batches as FRS | 650M | epoch with the lowest mean pairwise cosine |

Models 1–10 use the **primary mode (`strict`)** pair files. Models 11 and 12 differ from Model 1 only in the pair set (and its pool), Model 13 only in the MC sampling, Model 14 only in the checkpoint-selection rule.

**Why Model 14 exists.** The loss is lowest when every peptide sits at one point (alignment is rewarded, the uniformity term is weak), so 'lowest training loss among epochs whose guard is > 0.5' tends to select an early, nearly collapsed epoch. The collapse report prints `mean_cos` / `eff_rank` at each checkpoint; Model 14 selects on that spread directly so the two rules can be compared on the same pairs in Step 5.

Models 8 and 9 together give what the old notebook could not: with 8 vs 1 only the **backbone**
changes, and with 9 vs 6 only the **backbone** changes, so a scale effect is finally separable from a
pairing effect.

In [ ]:
print("=" * 70); print("1/15: Primary"); print("=" * 70)
run = train_model(primary_train, primary_val, seq_to_emb, DIM_650, "Primary", **CONFIG)
save_model(run, "model_primary.pt", "Primary", variant="all")

In [ ]:
print("=" * 70); print("2/15: HP1-Random"); print("=" * 70)
run = train_model(hp1_train, hp1_val, seq_to_emb, DIM_650, "HP1-Random", **CONFIG)
save_model(run, "model_hp1_random.pt", "HP1-Random", variant="random")

In [ ]:
print("=" * 70); print("3/15: HP2-Fusion"); print("=" * 70)
run = train_model(primary_train, primary_val, seq_to_fused_emb, DIM_FUSED, "HP2-Fusion", **CONFIG)
save_model(run, "model_hp2_fusion.pt", "HP2-Fusion", variant="fusion")

In [ ]:
print("=" * 70); print("4/15: Primary-Balanced"); print("=" * 70)
run = train_model(primary_train, primary_val, seq_to_emb, DIM_650, "Primary-Balanced",
                  mc_mode="balanced", **CONFIG)
save_model(run, "model_primary_balanced.pt", "Primary-Balanced", variant="all")

In [ ]:
print("=" * 70); print("5/15: ARI-Checkpoint-Only  (no floor -- kept as the E-013 negative control)"); print("=" * 70)
run = train_model(primary_train, primary_val, seq_to_emb, DIM_650, "ARI-Checkpoint-Only",
                  checkpoint_mode="ari", **CONFIG)
save_model(run, "model_ari_checkpoint_only.pt", "ARI-Checkpoint-Only", variant="all")

In [ ]:
print("=" * 70); print("6/15: k-NN-Pairing"); print("=" * 70)
run = train_model(knn_train, knn_val, seq_to_emb, DIM_650, "k-NN-Pairing", **CONFIG)
save_model(run, "model_knn_pairing.pt", "k-NN-Pairing", variant="knn")

In [ ]:
print("=" * 70); print("7/15: k-NN+ARI-Checkpoint"); print("=" * 70)
run = train_model(knn_train, knn_val, seq_to_emb, DIM_650, "k-NN+ARI-Checkpoint",
                  checkpoint_mode="ari", **CONFIG)
save_model(run, "model_knn_ari_checkpoint.pt", "k-NN+ARI-Checkpoint", variant="knn")

In [ ]:
print("=" * 70); print("8/15: Primary-150M  (N-9 FIX: primary_train, not knn_train)"); print("=" * 70)
# OLD (confounded): train_model(knn_train, knn_val, seq_to_emb_150m, 640, "Primary-150M")
run = train_model(primary_train, primary_val, seq_to_emb_150m, DIM_150, "Primary-150M", **CONFIG)
save_model(run, "model_primary_150m.pt", "Primary-150M", variant="all")

In [ ]:
print("=" * 70); print("9/15: k-NN-150M  (the old 'Primary-150M' artifact, correctly named)"); print("=" * 70)
run = train_model(knn_train, knn_val, seq_to_emb_150m, DIM_150, "k-NN-150M", **CONFIG)
save_model(run, "model_knn_150m.pt", "k-NN-150M", variant="knn")

In [ ]:
print("=" * 70); print("10/15: ARI-Checkpoint-Floored  (E-013 positive control)"); print("=" * 70)
run = train_model(primary_train, primary_val, seq_to_emb, DIM_650, "ARI-Checkpoint-Floored",
                  checkpoint_mode="ari", saved_dist_floor=ARI_FLOOR, **CONFIG)
save_model(run, "model_ari_checkpoint_floored.pt", "ARI-Checkpoint-Floored", variant="all")

In [ ]:
print("=" * 70); print("11/15: Primary-Extended  (sensitivity: extended mode -- candidate-evidence peptides included)"); print("=" * 70)
print("prototype/ARI pool:", len(POOLS["extended"]), "fold-0 peptides | train pairs:", len(ext_train), "| val pairs:", len(ext_val))
run = train_model(ext_train, ext_val, seq_to_emb, DIM_650, "Primary-Extended", proto_pool=POOLS["extended"], **CONFIG)
save_model(run, "model_primary_extended.pt", "Primary-Extended", variant="extended", mode="extended")

In [ ]:
print("=" * 70); print("12/15: Primary-StrictGE4  (sensitivity: strict evidence and >= 4 aa -- 3-aa peptides removed everywhere)"); print("=" * 70)
print("prototype/ARI pool:", len(POOLS["strict_ge4"]), "fold-0 peptides | train pairs:", len(ge4_train), "| val pairs:", len(ge4_val))
run = train_model(ge4_train, ge4_val, seq_to_emb, DIM_650, "Primary-StrictGE4", proto_pool=POOLS["strict_ge4"], **CONFIG)
save_model(run, "model_primary_strict_ge4.pt", "Primary-StrictGE4", variant="strict_ge4", mode="strict_ge4")

In [ ]:
print("=" * 70); print("13/15: Primary-MechEqual  (MC resampled to as many batches per epoch as FRS)"); print("=" * 70)
run = train_model(primary_train, primary_val, seq_to_emb, DIM_650, "Primary-MechEqual", mc_mode="equal", **CONFIG)
save_model(run, "model_primary_mechequal.pt", "Primary-MechEqual", variant="mech_equal")

In [ ]:
print("=" * 70); print("14/15: Primary-MaxSpread  (same pairs as Model 1; checkpoint = the epoch with the most spread-out embeddings)"); print("=" * 70)
run = train_model(primary_train, primary_val, seq_to_emb, DIM_650, "Primary-MaxSpread", checkpoint_mode="spread", **CONFIG, patience=40)
save_model(run, "model_primary_maxspread.pt", "Primary-MaxSpread", variant="max_spread")

In [ ]:
print("=" * 70); print("15/15: MechEqual-MaxSpread  (MC resampled to as many batches as FRS; checkpoint = the most spread-out epoch)"); print("=" * 70)
run = train_model(primary_train, primary_val, seq_to_emb, DIM_650, "MechEqual-MaxSpread", mc_mode="equal", checkpoint_mode="spread", **CONFIG, patience=40)
save_model(run, "model_mechequal_maxspread.pt", "MechEqual-MaxSpread", variant="mech_equal_max_spread")


---
## 5. Collapse report — the honest plateau count  *(N-4 fix)*

The v2.2 header claimed "the curve rises to a flat plateau and stays there". That is a per-model
property and it does not hold for every model: HP1-Random went **1.3854 at epoch 15 to 0.0274 at
epoch 18**. This cell counts, per model, how many epochs the **saved-prototype** distance stayed
above 0.5, and prints the qualifier as "**k of N**" instead of a blanket claim.

In [ ]:
rows = []
for rec in [r for r in SAVED if r["file"]]:
    ck = torch.load(f"{BASE}/results/{rec['file']}", map_location=device, weights_only=False)
    h = ck["history"]; ds = h["proto_dist_saved"]; n = len(ds)
    above = sum(1 for d in ds if d > 0.5)
    tail  = ds[-5:]
    rows.append({"model": rec["model"], "epochs": n,
                 "saved_peak": round(max(ds), 4), "saved_final": round(ds[-1], 4),
                 "epochs>0.5": f"{above}/{n}", "frac>0.5": round(above / n, 2),
                 "plateau_held": bool(min(tail) > 0.5),
                 "collapsed_at_end": bool(ds[-1] < 0.1),
                 "ckpt_saved_dist": rec["saved"], "FRS-GEN": rec["FRS-GEN"], "FRS-MC": rec["FRS-MC"],
                 "mean_cos_ckpt": round(ck["mean_cos_at_ckpt"], 3), "eff_rank_ckpt": round(ck["eff_rank_at_ckpt"], 1),
                 "mean_cos_final": round(h["mean_cos"][-1], 3), "eff_rank_final": round(h["eff_rank"][-1], 1),
                 "criterion_met": rec["criterion_met"]})

collapse = pd.DataFrame(rows)
print(collapse.to_string(index=False))

n_tot = len(collapse)
n_pl  = int(collapse["plateau_held"].sum())
n_col = int(collapse["collapsed_at_end"].sum())
n_gen = int((collapse["FRS-GEN"] < 0.1).sum())
print(f"\nHONEST SUMMARY (N-4):")
print(f"  plateau held through the last 5 epochs : {n_pl}/{n_tot}")
print(f"  saved min distance < 0.1 at final epoch: {n_col}/{n_tot}   (dominated by FRS-GEN; NOT a collapse test -- use mean_cos / eff_rank)")
print(f"  FRS and GEN merged in the SAVED model  : {n_gen}/{n_tot}   (descriptive: the head does not separate GEN peptides from FRS peptides; see 2b)")
print(f"  spread at the saved checkpoint         : mean_cos {collapse['mean_cos_ckpt'].min():.3f}..{collapse['mean_cos_ckpt'].max():.3f}, eff_rank {collapse['eff_rank_ckpt'].min():.1f}..{collapse['eff_rank_ckpt'].max():.1f}  (1 = one point / one direction)")
n_crit = int((~collapse["criterion_met"]).sum())
if n_crit:
    print(f"  *** {n_crit}/{n_tot} models met NO selection criterion and fell back to their")
    print(f"      best-separated epoch. Name them; do not compare them with the rest:")
    for m in collapse.loc[~collapse.criterion_met, "model"]:
        print(f"        - {m}")
print(f"\nWrite it as '{n_pl} of {n_tot} models hold a flat plateau', never as 'the recipe gives")
print("a flat plateau'. Name the exceptions.")

collapse.to_csv(f"{BASE}/results/step4_collapse_report.csv", index=False)
pd.DataFrame(SAVED).to_csv(f"{BASE}/results/step4_saved_models.csv", index=False)
print("\nSaved: results/step4_collapse_report.csv, results/step4_saved_models.csv")

### 5b. Did the distance floor work?  *(E-013 verdict)*

Model 5 (ARI, no floor) against Model 10 (ARI, `saved > 0.3`). If Model 10 keeps a separated
prototype set at a modest ARI cost, the project's own recommendation is **validated** and can be
stated as a result rather than a suggestion. If Model 10 cannot find any epoch above the floor, that
is also a finding: ARI and prototype separation are in direct conflict under this loss.


In [ ]:
cmp = pd.DataFrame(SAVED).set_index("model")
pair = [m for m in ["ARI-Checkpoint-Only", "ARI-Checkpoint-Floored"] if m in cmp.index]
print(cmp.loc[pair, ["epoch", "guard", "saved", "FRS-MC", "FRS-GEN", "GEN-MC", "ari_at_ckpt"]]
         .to_string())

if len(pair) == 2:
    a = cmp.loc["ARI-Checkpoint-Only"]; b = cmp.loc["ARI-Checkpoint-Floored"]
    print(f"\nsaved min distance : {a['saved']:.4f} -> {b['saved']:.4f}")
    print(f"ARI at checkpoint  : {a['ari_at_ckpt']:.4f} -> {b['ari_at_ckpt']:.4f}")
    if a["epoch"] == b["epoch"] and abs(a["saved"] - b["saved"]) < 1e-6:
        print(f"\nE-013 FLOOR NEVER BOUND: the unconstrained ARI checkpoint already sat above "
              f"{ARI_FLOOR}, so both models selected epoch {a['epoch']}. The floor is a no-op on "
              f"this run -- report it as a safeguard that was not needed here, NOT as a measured "
              f"improvement.")
    elif b["saved"] > ARI_FLOOR and b["ari_at_ckpt"] > 0:
        print(f"\nE-013 VALIDATED: the floor keeps the prototypes separated "
              f"(cost: {a['ari_at_ckpt'] - b['ari_at_ckpt']:+.4f} ARI).")
    else:
        print("\nE-013 NOT validated: no epoch satisfied both ARI improvement and the floor ->")
        print("report that ARI-based selection and prototype separation conflict under this loss.")

---
## 6. Verify the saved checkpoints, and prove nothing was overwritten  *(E-014 fix, asserted)*

The old notebook's `knn_pairs.to_csv(...)` meant Step 4 mutated Step 3's outputs. This cell asserts
the pair files have the **same SHA-256** as when they were loaded in Section 0b, so the claim "Step 3 is the
only producer" is checked rather than promised.

In [ ]:
EXPECT_FILES = [r["file"] for r in SAVED if r["file"]]
_skipped = [r["model"] for r in SAVED if not r["file"]]
if _skipped:
    print("MODELS WITH NO CHECKPOINT:", _skipped)
print("checkpoints:")
for fn in EXPECT_FILES:
    path = f"{BASE}/results/{fn}"
    assert os.path.exists(path), f"MISSING: {fn}"
    ck = torch.load(path, map_location=device, weights_only=False)
    assert {"head_state_dict", "prototypes", "min_proto_dist_saved", "pair_hashes", "s2_fingerprint", "pair_variant"} <= set(ck)
    assert ck["pair_hashes"] == PAIR_HASHES, f"{fn} was trained on different pair files"
    assert ck["s2_fingerprint"] == S2_FP
    print(f"  OK {fn:<38} dim={ck['input_dim']:<5} epoch={ck['checkpoint_epoch']:<3} variant={ck['pair_variant']:<7} "
          f"guard={ck['min_proto_dist_guard']:.3f} saved={ck['min_proto_dist_saved']:.3f}")

print("\nE-014 check -- input files unchanged by this notebook (SHA-256):")
for key, h in PAIR_HASHES.items():
    assert sha256_file(f"{PROC}/pairs/{key}") == h, f"E-014 REGRESSION: pairs/{key} changed during Step 4"
print(f"  all {len(PAIR_HASHES)} pair files unchanged")
for name, path in [("descriptors.csv", f"{PROC}/descriptors.csv"), ("esm2_650M.npy", f"{EMB}/esm2_650M.npy"), ("esm2_150M.npy", f"{EMB}/esm2_150M.npy")]:
    assert sha256_file(path) == EMB_SHA[name], f"{name} changed during Step 4"
print("  descriptors and embeddings unchanged")
print("\nStep 4 wrote only to results/. data/ is untouched.")

---
## Summary — what to carry into Step 5 and the thesis

**Artifacts** — 15 checkpoints in `results/` (each stamped with the S2 fingerprint, pair-file SHA-256, input hashes, pair variant and seed), plus `step4_collapse_report.csv`, `step4_saved_models.csv`, `anticollapse_sweep_fixed.png`.

**Claims this notebook supports, and the exact wording to use**

1. **Prototype separation must be quoted from `min_proto_dist_saved`, not `min_proto_dist_guard`.** The guard scores a GEN vector frozen at initialisation; the saved file holds a recomputed one (N-10). Any "collapse-free" statement based on the guard value is unsupported. **FRS–GEN ≈ 0 is descriptive, not a collapse criterion** (Section 2b); collapse is read from `mean_cos` / `eff_rank` in the collapse report.
2. **"k of N models hold a flat plateau"**, naming the exceptions — not "the recipe gives a flat plateau" (N-4).
3. **tau = 0.2 / lambda = 5.0 is a declared deviation** from the TAF's tau = 0.07 / lambda = 0.1, motivated by the 1/tau gradient argument (E-007). Whether it *works* is what the printed **SWEEP VERDICT** says; do not write "holds a separated plateau" unless that verdict supports it.
4. **Model 8 vs Model 1 is the scale comparison**; Model 9 `k-NN-150M` changes backbone *and* pairing and is not a scale test (N-9).
5. **E-013 has a verdict** from Model 10 rather than an open recommendation.
6. **Every model was trained on verified Step-3 v4 pairs** — the gates in Sections 0 and 0b stop the notebook on any fingerprint or SHA-256 mismatch, and the checkpoints carry them.
7. **Primary analysis = `strict` pairs (Models 1–10). Models 11 (`extended`) and 12 (`strict_ge4`) are sensitivity analyses built from their own complete pair sets** (own scaler/threshold, own k-NN, own random control). They are informative only if Step 5 evaluates them on the matching query and gallery sets (extended: all peptides; strict_ge4: strict peptides of ≥ 4 aa on both sides).
8. **Model 13 `Primary-MechEqual`** measures the effect of the ~50 : 1 FRS : MC row imbalance; any statement about MC should be made with Models 1, 4 and 13 side by side.

9. **Model 14 `Primary-MaxSpread`** differs from Model 1 only in the checkpoint-selection rule. If Model 1's checkpoint has `mean_cos` close to 1 while Model 14's does not, the lowest-loss rule selected a collapsed representation; Step 5 shows whether that matters for retrieval.

**Limits to state with the results**
* **One seed (42) and one fold direction** (train fold 0, validate fold 1). Differences between models smaller than the seed-to-seed spread are not evidence; Step 5 should report a seed spread for at least Primary and HP1-Random before drawing a conclusion on HP1.
* **The MC training signal is thin** (partner table in Section 1): a few hundred training pairs, a third of the strict MC peptides with no partner, a few tens of validation pairs. Say so next to any MC result.
* Checkpoint selection uses the training loss (or ARI on the training fold), not an independent held-out score.

**Next: Step 5** — load the checkpoints, assert `s2_fingerprint` and `pair_hashes`, evaluate with group CV by `homology_group`, add composition and length baselines, evaluate Models 11–12 on the matching subsets, and use the 27 clean negatives (`db_conflict == False`) from `AOP-Inactive.csv`.